In [2]:
# 1) Pip + setuptools
!python3 -m pip install --upgrade pip setuptools wheel
!python3 -m pip install 'setuptools<81'

# 2) Torch 2.6.0 + CUDA 12.4
!python3 -m pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
    --index-url https://download.pytorch.org/whl/cu124

# 3) Base deps
!python3 -m pip install numpy==2.1.1 scipy>=1.11.4 pyyaml opencv-python==4.11.0.86 \
  matplotlib==3.10.1 fvcore==0.1.5.post20221221 iopath==0.1.10 hydra-core==1.3.2 \
  tqdm einops omegaconf timm==0.9.12 networkx==3.1 --quiet

# 4) FlashAttention + YOLO
!python3 -m pip install flash-attn==2.7.4.post1 --no-build-isolation --no-deps
!python3 -m pip install ultralytics==8.3.99

# 5) Detectron2 + SAM family + CLIP
!python3 -m pip install --no-cache-dir --no-build-isolation --no-deps \
  git+https://github.com/facebookresearch/detectron2.git@main
!python3 -m pip install segment-anything-hq
!python3 -m pip install git+https://github.com/openai/CLIP.git@main
!python3 -m pip install git+https://github.com/facebookresearch/segment-anything.git@main
!python3 -m pip install --no-deps --no-build-isolation git+https://github.com/facebookresearch/sam2.git@main

# 6) Librerie NLP + HuggingFace + WBF & friends
!python3 -m pip install \
  spacy==3.8.4 nltk==3.9.1 blis \
  datasets==3.3.1 transformers>=4.50 sentence-transformers==3.4.1 \
  colorlog==6.9.0 pretty-errors==1.2.25 accelerate==1.4.0 \
  ensemble-boxes==1.0.7 adjustText==0.8 sentencepiece==0.2.0 \
  huggingface_hub[hf_xet]==0.31.1 num2words==0.5.13 \
  psutil==5.9.5 pycocotools pillow==10.2.0

# 7) Modelli SpaCy / NLTK
!python3 -m spacy download en_core_web_md
!python3 -m nltk.downloader wordnet

# 8) Verifica finale
import importlib, torch
print(f"✅ PyTorch {torch.__version__} | CUDA {torch.version.cuda} | GPU: {torch.cuda.is_available()}")
for pkg in ["detectron2","segment_anything","segment_anything_hq","sam2","clip","hydra","iopath","datasets","transformers"]:
    try:
        m = importlib.import_module(pkg)
        print(f"✅ {pkg}: OK")
    except Exception as e:
        print(f"⚠️ {pkg}: {e}")


Looking in indexes: https://download.pytorch.org/whl/cu124
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.35.2 requires huggingface-hub>=0.34.0, but you have huggingface-hub 0.31.1 which is incompatible.
gradio 5.49.1 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.31.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.1.1 which is incompatible.
nx-cugraph-cu12 25.10.0 requires networkx>=3.2, but you have networkx 3.1 which is incompatible.
  Cloning https://github.com/facebookresearch/detectron2.git (to revision main) to /tmp/pip-req-build-sknzj8pm
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-req-build-sknzj8pm
  Resolved https://github.com/facebookresearch/detectron2.git to commit a9c0821a12ad353fb2a96f019515990d5460c5ac
 

/usr/local/lib/python3.12/dist-packages/segment_anything_hq/modeling/tiny_vit_sam.py:662: UserWarning: Overwriting tiny_vit_5m_224 in registry with segment_anything_hq.modeling.tiny_vit_sam.tiny_vit_5m_224. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  return register_model(fn_wrapper)
/usr/local/lib/python3.12/dist-packages/segment_anything_hq/modeling/tiny_vit_sam.py:662: UserWarning: Overwriting tiny_vit_11m_224 in registry with segment_anything_hq.modeling.tiny_vit_sam.tiny_vit_11m_224. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  return register_model(fn_wrapper)
/usr/local/lib/python3.12/dist-packages/segment_anything_hq/modeling/tiny_vit_sam.py:662: UserWarning: Overwriting tiny_vit_21m_224 in registry with segment_anything_hq.modeling.tiny_vit_sam.tiny_vit_21m_224. This is because the name being registered conflicts with an existing name. Pl

✅ segment_anything_hq: OK
✅ sam2: OK
✅ clip: OK
✅ hydra: OK
✅ iopath: OK
✅ datasets: OK
✅ transformers: OK


In [ ]:
!unzip /content/graph-of-marks.zip

In [14]:
import sys, os
sys.path.append("/content/graph-of-marks/src")
os.chdir("/content/graph-of-marks")

from igp.pipeline.preprocessor import ImageGraphPreprocessor, PreprocessorConfig

cfg = PreprocessorConfig()

# =============================================================================
# 📥 INPUT / OUTPUT
# =============================================================================
cfg.input_path = "/content/living.png"
cfg.output_folder = "/content/gom_outputs"
cfg.output_format = "png"

cfg.save_image_only = False
cfg.skip_graph = False
cfg.skip_visualization = False
cfg.skip_prompt = False

# =============================================================================
# ❓ QUESTION-GUIDED FILTERING (mantenuti solo parametri utili)
# =============================================================================
cfg.question = "Where is the light?"
cfg.apply_question_filter = True
cfg.filter_relations_by_question = True

cfg.threshold_object_similarity = 0.40
cfg.threshold_relation_similarity = 0.30
cfg.clip_pruning_threshold = 0.20
cfg.semantic_boost_weight = 0.30

cfg.context_expansion_enabled = True
cfg.context_expansion_radius = 1.5
cfg.context_min_iou = 0.10

# =============================================================================
# 🔍 DETECTORS (solo parametri che contano davvero)
# =============================================================================
cfg.detectors_to_use = ["owlvit", "yolov8", "detectron2"]

cfg.threshold_owl = 0.35
cfg.threshold_yolo = 0.40
cfg.threshold_detectron = 0.40

cfg.max_detections_total = 200
cfg.min_detection_conf = 0.30

cfg.enable_detection_cache = True

cfg.detection_resize = True
cfg.detection_max_side = 1200   # HQ
cfg.detection_hash_method = "full"

# =============================================================================
# 🔄 FUSION / MERGE (solo quelli rilevanti)
# =============================================================================
cfg.wbf_iou_threshold = 0.55
cfg.label_nms_threshold = 0.60
cfg.seg_iou_threshold = 0.60

cfg.enable_group_merge = True
cfg.merge_mask_iou_threshold = 0.55
cfg.merge_box_iou_threshold = 0.75

cfg.enable_semantic_dedup = True
cfg.semantic_dedup_iou_threshold = 0.40

# =============================================================================
# 🎭 SEGMENTATION — HQ SAM-HQ
# =============================================================================
cfg.sam_version = "hq"
cfg.sam_hq_model_type = "vit_h"
cfg.points_per_side = 64

cfg.pred_iou_thresh = 0.90
cfg.stability_score_thresh = 0.95

# =============================================================================
# 🎨 VISUALIZATION — essenziali
# =============================================================================
cfg.display_labels = True
cfg.display_relationships = True
cfg.display_relation_labels = True

cfg.show_segmentation = True
cfg.fill_segmentation = True
cfg.seg_fill_alpha = 0.25

cfg.display_legend = False
cfg.resolve_overlaps = True

cfg.show_bboxes = True
cfg.bbox_linewidth = 2.0

cfg.color_sat_boost = 1.1
cfg.color_val_boost = 1.1

# =============================================================================
# ⚙️ SYSTEM
# =============================================================================
cfg.preproc_device = "cuda"
cfg.verbose = True

# =============================================================================
# 🚀 RUN
# =============================================================================
pre = ImageGraphPreprocessor(cfg)
pre.run()


YOLOv8x summary (fused): 112 layers, 68,200,608 parameters, 0 gradients, 257.8 GFLOPs
<All keys matched successfully>


INFO:igp.pipeline.preprocessor:
INFO:igp.pipeline.preprocessor:======================================================================
INFO:igp.pipeline.preprocessor:🖼️  Processing: living
INFO:igp.pipeline.preprocessor:📐 Size: 1056x992 px
INFO:igp.pipeline.preprocessor:❓ Question: Where is the light?
INFO:igp.pipeline.preprocessor:======================================================================
INFO:igp.pipeline.preprocessor:
🔍 [1/7] Object Detection
INFO:igp.pipeline.preprocessor:   ⚙️  Running detectors...


[RelationInferencer] PhysicsReasoner activated


INFO:igp.pipeline.preprocessor:   ✅ Detected 34 objects
INFO:igp.pipeline.preprocessor:   📊 Top objects: clock(0.96), sofa(0.93), sofa(0.90), chair(0.89), vase(0.88)
INFO:igp.pipeline.preprocessor:
🔎 [2/7] Question-Based Filtering
INFO:igp.pipeline.preprocessor:
✂️  [3/7] Pruning & NMS
INFO:igp.pipeline.preprocessor:   ✅ 34 → 34 objects (removed 0 duplicates/low-score)
INFO:igp.pipeline.preprocessor:
🎭 [4/7] Segmentation (SAM)
INFO:igp.pipeline.preprocessor:   ⚙️  Generating masks for 34 objects...



[DEBUG SINGLETON CHECK]
  Question: 'Where is the light?'
  obj_terms: {'alight', 'Inner Light', 'the light', 'live', 'embody', 'lightheaded', 'weak', 'cost', 'easy', 'comprise', 'visible light', 'twinkle', 'abstemious', 'lightly', 'ignitor', 'brightness level', 'wakeful', 'unhorse', 'lighter', 'promiscuous', 'faint', 'be', 'Christ Within', 'illume', 'clear', 'exist', 'Light', 'get down', 'low-cal', 'light', 'igniter', 'get off', 'constitute', 'clean', 'follow', 'light-headed', 'swooning', 'lighting', 'illuminate', 'unaccented', 'light source', 'lite', 'Light Within', 'loose', 'spark', 'ignite', 'equal', 'is the light', 'scant', 'fall', 'is_the_light', 'illumine', 'calorie-free', 'luminousness', 'light-colored', 'brightness', 'unclouded', 'perch', 'personify', 'luminance', 'visible radiation', 'idle', 'the_light', 'light up', 'sluttish', 'lightness', 'represent', 'illumination', 'sparkle', 'short', 'wanton', 'lightsome', 'make up', 'dismount', 'luminosity', 'tripping', 'fire up'}
  ap

INFO:igp.pipeline.preprocessor:   ✅ Generated 34 segmentation masks
INFO:igp.pipeline.preprocessor:
📏 [5/7] Depth Estimation
INFO:igp.pipeline.preprocessor:   ⚙️  Computing depth for 31 objects...



🧹 [4.5/7] Post-Segmentation Deduplication
   Checking for overlapping objects...
  [DEDUP] Removed table (score=0.285, overlaps sofa score=0.901, IoU=0.181, score_diff=0.684)
  [DEDUP] Removed table (score=0.183, overlaps chair score=0.889, IoU=0.116, score_diff=0.794)
  [DEDUP] Removed bird (score=0.573, overlaps light score=0.426, i 94.6% contained in TARGET j (IoU=0.119))
  [DEDUP] Removed 3 overlapping objects (34 → 31)
   ✅ Removed 3 overlapping objects
[DEPTH] ✓ Reusing cached model: depth_anything_v2_vitl_cuda_torch.float16


INFO:igp.pipeline.preprocessor:   ✅ Depth computed
INFO:igp.pipeline.preprocessor:
🔗 [6/7] Spatial Relations Inference
INFO:igp.pipeline.preprocessor:   ⚙️  Analyzing relationships between 31 objects...
INFO:igp.pipeline.preprocessor:[REL-OPT] Pruned 1 objects; using 30 for relation inference
INFO:igp.pipeline.preprocessor:   ✅ Found 422 candidate relationships
INFO:igp.pipeline.preprocessor:[SINGLETON] Filtered 31 → 8 objects (target + connected)
INFO:igp.pipeline.preprocessor:[SINGLETON]   Target objects: [24]
INFO:igp.pipeline.preprocessor:[SINGLETON]   Connected objects: [0, 1, 3, 7, 25, 27, 28]
INFO:igp.pipeline.preprocessor:[SINGLETON]   Relations: 138 → 7
INFO:igp.pipeline.preprocessor:
🎨 [7/7] Visualization & Export
INFO:igp.pipeline.preprocessor:   ⚙️  Rendering: segmentation, relationships, labels, bboxes
INFO:igp.pipeline.preprocessor:   📁 Format: PNG



🎯 [SINGLETON FILTERING] Target + Connected Objects Only (post-relation-filter)
   Target indices: [24]
   Target labels: ['light']
   Total objects before filter: 31
   Total relations before filter: 138
   Connected object indices: [0, 1, 3, 7, 25, 27, 28]
   Connected labels: ['clock', 'sofa', 'chair', 'book', 'picture', 'curtain', 'potted plant']
   Calling _filter_objects_keep_target_and_connected...
   ✅ After filter: 8 objects, 7 relations
   Kept objects: ['clock', 'sofa', 'chair', 'book', 'light', 'picture', 'curtain', 'potted plant']


INFO:igp.pipeline.preprocessor:   ✅ Saved: /content/gom_outputs/living_output.png
INFO:igp.pipeline.preprocessor:
INFO:igp.pipeline.preprocessor:✅ Preprocessing complete!
INFO:igp.pipeline.preprocessor:⏱️  Total time: 28.52s
INFO:igp.pipeline.preprocessor:📊 Final results:
INFO:igp.pipeline.preprocessor:   • Objects: 8
INFO:igp.pipeline.preprocessor:   • Relationships: 15
INFO:igp.pipeline.preprocessor:   • Segmentation masks: 8
INFO:igp.pipeline.preprocessor:======================================================================
INFO:igp.pipeline.preprocessor:


In [ ]:
!cd /content/graph-of-marks && \
python3 src/image_preprocessor.py \
  --input_path "/content/Gemini_Generated_Image_kmy384kmy384kmy3.png" \
  --output_folder "/content/gom_outputs" \
  --detectors "owlvit,yolov8,detectron2" \
  --question "Where is the light?" \
  --display_labels \
  --show_segmentation \
  --fill_segmentation
